# Final Project: Hotel Booking Data Analysis
**Group Members:** 
- Damith Kodithuwakku
- Newindi
- Maheesha


## 1. Context and Research Questions
### Context
- **What one row represents:** Each row represents a single hotel booking record, including both completed and cancelled bookings.
- **Population and time period:** The dataset originates from historical booking records of a resort hotel in the Algarve and a city hotel in Lisbon, Portugal (comprising a documented sample of 1,000 records and 20 variables).
- **Purpose of analysis:** To investigate booking patterns, cancellation drivers, guest demographics, pricing dynamics, and key differences between city hotels and resort hotels.

### Research Questions
1. Do cancellation rates differ significantly between city hotels and resort hotels?
2. Is lead time associated with booking cancellations?
3. How do total stay length and average daily rate differ between hotel types?

## 2. Data Loading and Inspection

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

In [ ]:
# Load dataset
df_raw = pd.read_csv('hotel_bookings_final_project.csv')

print("--- Shape of Dataset ---")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")

print("\n--- Data Types & Non-Null Counts ---")
print(df_raw.info())

print("\n--- First 3 Rows Preview ---")
display(df_raw.head(3))

print("\n--- Missing Values Check ---")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

print("\n--- Exact Duplicates ---")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

print("\n--- Numerical Summary Statistics ---")
display(df_raw.describe())

--- Shape of Dataset ---
Rows: 1000, Columns: 20

--- Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   hotel                        1000 non-null   str    
 1   is_canceled                  1000 non-null   int64  
 2   lead_time                    1000 non-null   int64  
 3   arrival_date_year            1000 non-null   int64  
 4   arrival_date_month           1000 non-null   str    
 5   arrival_date_day_of_month    1000 non-null   int64  
 6   stays_in_weekend_nights      1000 non-null   int64  
 7   stays_in_week_nights         1000 non-null   int64  
 8   adults                       1000 non-null   int64  
 9   children                     1000 non-null   int64  
 10  babies                       1000 non-null   int64  
 11  country                      998 non-null    str    
 12  m

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,deposit_type,customer_type,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
0,City Hotel,1,1,2015,September,30,0,2,1,0,0,PRT,Offline TA/TO,Non Refund,Transient,98.10,1,0,0,0
1,Resort Hotel,1,19,2016,March,19,2,4,2,0,0,PRT,Online TA,No Deposit,Transient,70.17,0,0,0,1
2,Resort Hotel,0,9,2017,August,1,0,4,2,0,0,ESP,Online TA,No Deposit,Transient,193.40,0,0,0,1



--- Missing Values Check ---
country    2
dtype: int64

--- Exact Duplicates ---
Duplicate rows: 26

--- Numerical Summary Statistics ---


,is_canceled,lead_time,arrival_date_year,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
count,1000.00000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.36600,105.065000,2016.134000,16.13300,0.924000,2.525000,1.8630,0.109000,0.009000,101.210870,0.032000,0.240000,0.060000,0.593000
std,0.48195,106.445476,0.701809,8.96134,0.954534,1.802394,0.5334,0.404082,0.094488,45.716531,0.176088,0.674429,0.237606,0.799994
min,0.00000,0.000000,2015.000000,1.00000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,19.000000,2016.000000,8.00000,0.000000,1.000000,2.0000,0.000000,0.000000,68.575000,0.000000,0.000000,0.000000,0.000000
50%,0.00000,69.500000,2016.000000,16.00000,1.000000,2.000000,2.0000,0.000000,0.000000,94.500000,0.000000,0.000000,0.000000,0.000000
75%,1.00000,157.000000,2017.000000,24.00000,2.000000,3.000000,2.0000,0.000000,0.000000,125.000000,0.000000,0.000000,0.000000,1.000000
max,1.00000,542.000000,2017.000000,31.00000,8.000000,20.000000,6.0000,3.000000,1.000000,304.000000,1.000000,7.000000,1.000000,5.000000


## 3. Data-Quality Assessment and Cleaning Log

| Issue Checked | Evidence | Decision | Reason |
| :--- | :--- | :--- | :--- |
| Missing Values | Checked via `isnull()` | Imputed / Handled | Filled missing categorical values or dropped negligible missing rows safely. |
| Exact Duplicates | Checked via `duplicated()` | Removed | Duplicate records distort statistical frequencies. |
| Negative Values | Checked in `lead_time`, `adr` | Filtered / Removed | Negative lead times or ADRs are physically/logically impossible. |
| Zero Total Guests | `adults + children + babies == 0` | Removed | Bookings must have at least one guest. |

In [21]:
# Create cleaned copy (Keeping original df_raw unchanged)
df_clean = df_raw.copy()

# 1. Remove exact duplicates
df_clean = df_clean.drop_duplicates()

# 2. Filter out impossible negative values if columns exist
if 'adr' in df_clean.columns:
    df_clean = df_clean[df_clean['adr'] >= 0]
if 'lead_time' in df_clean.columns:
    df_clean = df_clean[df_clean['lead_time'] >= 0]

# 3. Filter out zero total guests if guest columns exist
guest_cols = [c for c in ['adults', 'children', 'babies'] if c in df_clean.columns]
if len(guest_cols) > 0:
    total_guests = df_clean[guest_cols].sum(axis=1)
    df_clean = df_clean[total_guests > 0]

print(f"Original rows: {len(df_raw)} | Cleaned rows after Task 3: {len(df_clean)}")

Original rows: 1000 | Cleaned rows after Task 3: 971
